# Lesson 2: How GPT-2 turns text into token IDs

GPT-1's Tiny Shakespeare example gave every character its own ID. GPT-2 uses a byte-level BPE tokenizer: it first splits text into pieces, represents bytes with a reversible byte-to-symbol mapping, then merges byte pieces according to learned ranks. The result is a sequence of integers that can index the model's embedding table.

We will use OpenAI's `tiktoken` library for the real GPT-2 encoding and a tiny example to make the merge rule visible. C and Java use the same GPT-2 vocabulary so we can compare IDs directly. No model weights, dataset download, or GPU are needed.

Reference code: [`train_gpt2.py`](../../build-nanogpt/train_gpt2.py) selects the GPT-2 encoding; [`fineweb.py`](../../build-nanogpt/fineweb.py) adds an end-of-text token between documents.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "gpt-2", *Path.cwd().parents]
tutorial_root = next(
    (candidate for candidate in candidates
     if (candidate / "experiments/configs/tiny-playground.json").is_file()),
    None,
)
if tutorial_root is None:
    raise FileNotFoundError("Open the repository, gpt-2 folder, or its notebook folder and restart the kernel.")

from gpt2_lab.config import ModelConfig
import tiktoken

encoding = tiktoken.get_encoding("gpt2")
gpt2 = ModelConfig.gpt2_124m()
print(f"GPT-2 encoding: {encoding.name}; vocabulary entries: {gpt2.vocab_size:,}")
print(f"Kernel: {sys.executable}")
print(f"Lesson project: {tutorial_root}")

## 1. From characters to subword tokens

A character tokenizer maps one character to one ID. GPT-2's tokenizer can represent a common word or word fragment with one ID, while unusual text can take several. Spaces are part of the input too, so `"hello"` and `" hello"` need not have the same tokenization.

The model does not receive the printed word pieces. It receives integer IDs, then looks up one vector per ID. The notebook prints each token's bytes in hexadecimal because an individual token can end in the middle of a multi-byte UTF-8 character.

In [ ]:
def inspect_text(text: str) -> list[int]:
    token_ids = encoding.encode_ordinary(text)
    print(f"Text: {ascii(text)}")
    print(f"UTF-8 bytes ({len(text.encode('utf-8'))}): {text.encode('utf-8').hex(' ')}")
    print(f"Token IDs ({len(token_ids)}): {token_ids}")
    print(f"Decoded as a whole: {ascii(encoding.decode(token_ids))}")
    print("One row per token: (ID, token bytes, UTF-8 fragment with non-ASCII escaped)")
    for token_id in token_ids:
        token_bytes = encoding.decode_single_token_bytes(token_id)
        fragment = ascii(token_bytes.decode('utf-8', errors='backslashreplace'))
        print(f"  {token_id:>5}  {token_bytes.hex(' '):<26} {fragment}")
    print()
    return token_ids

hello_ids = inspect_text("Hello, world!")

The UTF-8 bytes are losslessly mapped to a safe set of visible symbols before BPE operates. The symbols shown by a tokenizer debugger can look odd; the IDs and their byte sequence are the useful facts. Decoding the complete ID sequence puts the bytes back together and recovers the original text.

A BPE tokenizer has two ideas inside it: a regular-expression pre-tokenizer keeps certain text boundaries intact, then ranked pair merges combine byte pieces. GPT-2's real vocabulary contains tens of thousands of learned entries, so we will use a tiny made-up rank table to see the merge loop without reimplementing that full vocabulary.

In [ ]:
def merge_by_rank(symbols: list[str], ranks: dict[tuple[str, str], int]) -> list[str]:
    symbols = symbols.copy()
    while len(symbols) > 1:
        ranked_pairs = [
            (ranks[pair], index)
            for index, pair in enumerate(zip(symbols, symbols[1:]))
            if pair in ranks
        ]
        if not ranked_pairs:
            break
        _, index = min(ranked_pairs)
        symbols[index:index + 2] = [symbols[index] + symbols[index + 1]]
    return symbols

toy_ranks = {("l", "o"): 0, ("lo", "w"): 1, ("e", "r"): 2}
print(merge_by_rank(list("lower"), toy_ranks))
print("This toy table illustrates ranked merging; GPT-2 uses its own byte-level vocabulary and ranks.")

## 2. Compare spaces and unfamiliar text

Run the same inspection on English, Spanish punctuation, and an emoji. Compare `hello` with ` hello`. Then relate the number of IDs to the context window: GPT-2's 1,024 positions count tokens, not characters or words. The token count depends on the text and this specific vocabulary.

In [ ]:
examples = [
    "hello",
    " hello",
    "Hola, ¿qué tal?",
    "A robot wrote a poem 🤖",
]
for example in examples:
    inspect_text(example)

## 3. Mark document boundaries with end-of-text

`<|endoftext|>` is a special GPT-2 token with ID 50,256. In the FineWeb preparation code, Karpathy prepends this ID to each document before joining the token sequences. This lets training examples cross from one document into the next with an explicit boundary marker.

The marker is not ordinary text. The library's safe default rejects a literal special-token spelling unless the caller explicitly allows it. Here we request it deliberately, then add it to a short document to see the resulting sequence.

In [ ]:
end_of_text = encoding.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})
document = "The dragon learned Python."
document_ids = encoding.encode_ordinary(document)
joined_document = end_of_text + document_ids
print(f"Special token: {end_of_text}")
print(f"Document token IDs: {document_ids}")
print(f"Boundary + document: {joined_document}")
print(f"Round trip after removing boundary: {ascii(encoding.decode(joined_document[1:]))}")

sample_path = tutorial_root / "data" / "samples" / "playground.txt"
sample_documents = [line for line in sample_path.read_text(encoding="utf-8").splitlines() if line.strip()]
encoded_documents = [encoding.encode_ordinary(text) for text in sample_documents]
dataset_ids = [token_id for document_ids in encoded_documents for token_id in [*end_of_text, *document_ids]]
print(f"Toy dataset: {len(sample_documents)} documents, {sum(map(len, sample_documents)):,} characters")
print(f"Token IDs including one boundary per document: {len(dataset_ids):,}")
print(f"First 24 IDs: {dataset_ids[:24]}")

## Compare C and Java

Both companions use the GPT-2 vocabulary and print the same sample IDs. Java uses JTokkit's `r50k_base` encoding. C calls the small C API from `tiktoken-c`; its opaque tokenizer pointer and matching `tiktoken_free` calls make the library's ownership rules visible. The C dependency is downloaded by the setup script and kept out of Git.

From the repository root, run one step or the complete lesson:

```powershell
.\scripts\run-gpt2.ps1 -Language c -Action lesson -Lesson 2 -Part 1
.\scripts\run-gpt2.ps1 -Language java -Action lesson -Lesson 2 -Part all
```

The Python notebook is the explanation and interactive scratchpad. C and Java use ordinary functions and a class, while preserving the same three conceptual steps.

Library references: [OpenAI tiktoken](https://github.com/openai/tiktoken), [JTokkit](https://github.com/knuddelsgmbh/jtokkit), and [tiktoken-c](https://github.com/kojix2/tiktoken-c).

## Try an experiment

Choose a line from `gpt-2/data/samples/playground.txt`, encode it, and compare character count, UTF-8 byte count, and token count. Try adding a leading space, changing punctuation, or replacing a word with an emoji. Then explain why the same text must use the same tokenizer when its IDs are used to train or run a GPT-2 checkpoint.

Next we will use token IDs as a dataset: inspect how sequences are divided into train and validation data and how batches of next-token examples are formed.